# ART SLIDE｜世界の有名絵画を約100点追加

モナリザを含む177件の著名絵画候補から、現在の作品データと重複しない作品を最大100点取得します。AIC/Metの画像サーバーは使わず、**Wikidataで作品情報を照合し、Wikimedia Commonsの各ファイルのパブリックドメイン情報を確認**します。

**手順**：Colab Secretsに`GITHUB_TOKEN`（ryouy/slidesのContents Read/Write）を登録。解説文をOpenAIで生成する場合は`OPENAI_API_KEY`も登録。すべてのセルを実行します。

著作権が確認できないファイル、作者没後71年未満、1929年より新しい制作年、既存作品、低解像度画像は除外します。100点取得を保証するものではありません。公開前に作品の正確さと個別の権利表示を目視確認してください。


In [ ]:
%pip -q install requests pillow

In [ ]:
# ART SLIDE: curated famous paintings collector (Wikidata + Wikimedia Commons).
import base64, io, json, re, time, random, unicodedata, hashlib
from collections import Counter
from datetime import date
from urllib.parse import urlparse
import requests
from PIL import Image, ImageOps
from google.colab import userdata

REPO="ryouy/slides"
BASE="src/games/art-slide/data/artworks.ts"
EXTRA="src/games/art-slide/data/additional-artworks.ts"
RESEARCH="src/games/art-slide/data/artworks-research.json"
TARGET=100
FAMOUS_TITLES=["Mona Lisa","The Starry Night","The Scream","Girl with a Pearl Earring","The Birth of Venus","The Creation of Adam","The Last Supper (Leonardo)","The School of Athens","The Night Watch","The Anatomy Lesson of Dr. Nicolaes Tulp","The Return of the Prodigal Son (Rembrandt)","The Jewish Bride","The Milkmaid (Vermeer)","The Art of Painting","The Astronomer (Vermeer)","The Geographer","The Little Street","View of Delft","The Lacemaker (Vermeer)","The Arnolfini Portrait","The Garden of Earthly Delights","The Haywain Triptych","The Tower of Babel (Bruegel)","The Peasant Wedding","The Hunters in the Snow","The Blind Leading the Blind","The Triumph of Death","Las Meninas","The Surrender of Breda","The Rokeby Venus","The Third of May 1808","Saturn Devouring His Son","The Nude Maja","Liberty Leading the People","The Raft of the Medusa","The Death of Marat","The Coronation of Napoleon","Napoleon Crossing the Alps","Oath of the Horatii","Grande Odalisque","The Fighting Temeraire","The Slave Ship","Rain, Steam and Speed – The Great Western Railway","The Hay Wain","Salisbury Cathedral from the Meadows","Wanderer above the Sea of Fog","The Sea of Ice","The Monk by the Sea","The Kiss (Klimt)","Judith and the Head of Holofernes (Klimt)","The Great Wave off Kanagawa","Fine Wind, Clear Morning","Thunderstorm Beneath the Summit","Sudden Shower over Shin-Ōhashi bridge and Atake","Plum Park in Kameido","The Card Players","Mont Sainte-Victoire (Cézanne)","The Bathers (Cézanne)","The Basket of Apples","Olympia (Manet)","A Bar at the Folies-Bergère","The Luncheon on the Grass","The Fifer","The Railway (Manet)","Impression, Sunrise","Woman with a Parasol","Water Lilies (Monet)","The Japanese Footbridge","La Grenouillère (Monet)","The Magpie (Monet)","Poppies (Monet)","The Artist's Garden at Giverny","Bal du moulin de la Galette","Luncheon of the Boating Party","The Umbrellas (Renoir)","Two Sisters (On the Terrace)","Dance at Bougival","The Dance Class","The Star (Degas)","L'Absinthe","The Bellelli Family","The Bedroom","Irises (painting)","Café Terrace at Night","The Night Café","Almond Blossoms","Wheatfield with Crows","Starry Night Over the Rhône","The Potato Eaters","Self-Portrait with Bandaged Ear","The Red Vineyard","The Yellow House","The Church at Auvers","Where Do We Come From? What Are We? Where Are We Going?","Tahitian Women on the Beach","The Spirit of the Dead Watching","Vision After the Sermon","The Yellow Christ","When Will You Marry?","The Gross Clinic","The Agnew Clinic","Whistler's Mother","Symphony in White, No. 1: The White Girl","Nocturne in Black and Gold – The Falling Rocket","Carnation, Lily, Lily, Rose","Madame X (portrait)","The Daughters of Edward Darley Boit","El Jaleo","The Gulf Stream (painting)","Snap the Whip (painting)","Breezing Up (A Fair Wind)","The Fog Warning","Eight Bells (painting)","The Herring Net","The Veteran in a New Field","The Old Guitarist","Les Demoiselles d'Avignon","Family of Saltimbanques","The Dance (Matisse)","The Red Studio","Woman with a Hat","A Sunday Afternoon on the Island of La Grande Jatte","Bathers at Asnières","The Circus (Seurat)","The Sleeping Gypsy","The Dream (Rousseau painting)","Tiger in a Tropical Storm","Nighthawks (painting)","Automat (painting)","American Gothic","Christina's World","The Gleaners","The Angelus (painting)","The Stone Breakers","A Burial at Ornans","The Horse Fair","Ophelia (painting)","The Lady of Shalott (painting)","The Awakening Conscience","The Ambassadors (Holbein)","The Blue Boy","Mr and Mrs Andrews","The Swing (Fragonard)","The Embarkation for Cythera","The Death of Socrates","The Death of Sardanapalus","The Massacre at Chios","The Women of Algiers","The Wedding at Cana (Veronese)","The Tempest (Giorgione)","Sleeping Venus (Giorgione)","Venus of Urbino","Bacchus and Ariadne","Sacred and Profane Love","The Garden of Love (Rubens)","The Elevation of the Cross (Rubens)","The Descent from the Cross (Rubens)","The Three Graces (Rubens)","Et in Arcadia ego","The Burial of Count Orgaz","View of Toledo","The Calling of Saint Matthew","Judith Beheading Holofernes (Caravaggio)","The Entombment of Christ (Caravaggio)","The Cardsharps","The Fortune Teller (Caravaggio)","The Concert (Vermeer)","The Procuress (Vermeer)","The Woman Holding a Balance","The Laughing Cavalier","The Nightmare (Fuseli)","The Death of General Wolfe","The Heart of the Andes","The Oxbow","Among the Sierra Nevada, California","Kindred Spirits (painting)","The Ninth Wave"]
UA="ART-SLIDE-famous-art-collector/1.0 (research and noncommercial-access; Colab)"
WIKI="https://en.wikipedia.org/w/api.php"
WD="https://www.wikidata.org/w/api.php"
COMMONS="https://commons.wikimedia.org/w/api.php"
gh=requests.Session()
gh.headers.update({"Authorization":"Bearer "+(userdata.get("GITHUB_TOKEN") or ""),"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","User-Agent":UA})
if gh.headers["Authorization"]=="Bearer ": raise RuntimeError("Colab Secrets: GITHUB_TOKEN required")
openai_key=(userdata.get("OPENAI_API_KEY") or "").strip()
api=requests.Session()
api.headers.update({"User-Agent":UA,"Accept":"application/json"})
def get_json(url,params=None):
    for attempt in range(5):
        time.sleep(0.8)
        r=api.get(url,params=params,timeout=70)
        if r.status_code in (429,500,502,503,504):
            delay=min(65,4*2**attempt)+random.random()
            print("API rate-limited; waiting",round(delay),"seconds")
            time.sleep(delay)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("API unavailable; stop, don't flood server: "+url)
def gh_api(method,path,payload=None):
    url="https://api.github.com/repos/"+REPO+path
    for attempt in range(4):
        r=gh.request(method,url,json=payload,timeout=110)
        if r.status_code in (429,502,503,504):
            time.sleep(3*2**attempt)
            continue
        if r.status_code>=400: raise RuntimeError(f"GitHub {r.status_code}: {r.text[:250]}")
        return r.json()
    raise RuntimeError("GitHub request failed")
def repo_text(path,sha):
    r=gh.get("https://api.github.com/repos/"+REPO+"/contents/"+path,params={"ref":sha},timeout=70)
    if r.status_code==404: return ""
    r.raise_for_status()
    return base64.b64decode(r.json()["content"]).decode("utf-8")
def read_artworks(path,variable,sha):
    source=repo_text(path,sha)
    if not source: return []
    match=re.search(r"export const "+variable+r": Artwork\[\]\s*=\s*(\[.*?\]);",source,re.S)
    if not match: raise RuntimeError("Cannot parse "+path+"; avoiding duplicate imports")
    return json.loads(match.group(1))
def normalized(s):
    s=unicodedata.normalize("NFKD",str(s or "")).casefold()
    s="".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+"," ",s).strip()
def first_claim(item,p):
    for x in item.get("claims",{}).get(p,[]):
        val=x.get("mainsnak",{}).get("datavalue",{}).get("value")
        if val is not None:return val
    return None
def ent_id(v):
    return v.get("id") if isinstance(v,dict) else None
def year(v):
    match=re.search(r"^\+?(\d{4})-",v.get("time","")) if isinstance(v,dict) else None
    return int(match.group(1)) if match else None
def label(v):
    return v.get("labels",{}).get("en",{}).get("value","")
def entities(ids):
    out={}
    for start in range(0,len(ids),35):
        data=get_json(WD,{"action":"wbgetentities","format":"json","ids":"|".join(ids[start:start+35]),"props":"labels|claims","languages":"en"})
        out.update(data.get("entities",{}))
    return out
def image_meta(filename):
    pages=get_json(COMMONS,{"action":"query","format":"json","formatversion":"2","prop":"imageinfo","titles":"File:"+filename,"iiprop":"url|size|extmetadata|sha1","iiurlwidth":1200}).get("query",{}).get("pages",[])
    if not pages or not pages[0].get("imageinfo"):return None
    info=pages[0]["imageinfo"][0]
    ext=info.get("extmetadata",{})
    def field(key):
        return re.sub(r"<[^>]*>","",str(ext.get(key,{}).get("value",""))).strip()
    lic=field("LicenseShortName")
    license_url=field("LicenseUrl")
    usage=field("UsageTerms")
    copyright_flag=field("Copyrighted").lower()
    is_pd=("public domain" in lic.lower() or lic.lower().startswith(("pd-","cc0")) or "public domain" in usage.lower() or "creativecommons.org/publicdomain/zero" in license_url)
    if not is_pd or copyright_flag in ("true","yes","1"):return None
    url=info.get("thumburl") or info.get("url")
    if urlparse(url or "").hostname!="upload.wikimedia.org":return None
    return {"url":url,"page":info.get("descriptionurl",""),"sha1":info.get("sha1",""),"license":lic or "Public domain","licenseUrl":license_url or "https://commons.wikimedia.org/wiki/Commons:Licensing"}
def bilingual(title,artist,yr):
    basic=f"{title} is a painting by {artist}, dated {yr}. The artwork is documented in Wikidata and Wikimedia Commons."
    ja=f"「{title}」は{artist}が{yr}年に制作した絵画です。作品情報はWikidata、画像はWikimedia Commonsの公開資料に基づきます。"
    if not openai_key:return basic,ja
    try:
        r=requests.post("https://api.openai.com/v1/chat/completions",headers={"Authorization":"Bearer "+openai_key},json={
            "model":"gpt-4o-mini","messages":[{"role":"system","content":"Art catalogue editor. Do not invent specific visual details, provenance or unverified claims. Respond only JSON with english and japanese strings."},{"role":"user","content":f"Write accessible informative EN and JA captions, ideally English 250-550 chars and Japanese 120-300 chars; limits are soft. Verified facts: Painting {title}; creator {artist}; year {yr}. If facts are too sparse, keep concise."}],"response_format":{"type":"json_object"},"temperature":0.2,"max_tokens":700},timeout=90)
        r.raise_for_status()
        d=json.loads(r.json()["choices"][0]["message"]["content"])
        en=str(d.get("english") or "").strip()
        jp=str(d.get("japanese") or "").strip()
        return (en,jp) if en and jp else (basic,ja)
    except Exception as e:
        print("Description fallback:",str(e)[:90])
        return basic,ja
def era(y):
    if y<1700:return "Before 1700"
    if y<1800:return "1700s"
    if y<1850:return "1800–1849"
    if y<1880:return "1850–1879"
    if y<1900:return "1880–1899"
    return "1900s"
def theme(t):
    t=normalized(t)
    if any(x in t for x in ("wave","sea","ship","water","river","coast")):return "Sea & Water"
    if any(x in t for x in ("garden","landscape","mountain","snow","forest","field","wheat")):return "Landscape"
    if any(x in t for x in ("portrait","woman","girl","boy","mother","sisters")):return "Portrait"
    if any(x in t for x in ("flower","sunflower","lilies","still life","fruit")):return "Still Life & Flowers"
    if any(x in t for x in ("saint","christ","madonna","angel","judith","last supper")):return "Myth & Religion"
    if any(x in t for x in ("street","cafe","city","paris","bridge")):return "City & Street"
    return "People & Daily Life"
def convert_image(data):
    im=ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
    w,h=im.size
    if min(w,h)<350 or not (0.35<=w/h<=2.8):raise ValueError("Image resolution/aspect rejected")
    result=[]
    for px,quality in ((1200,82),(420,76)):
        resized=im.copy()
        resized.thumbnail((px,px),Image.Resampling.LANCZOS)
        b=io.BytesIO()
        resized.save(b,"WEBP",quality=quality,method=5)
        result.append((b.getvalue(),resized.size))
    return result
head=gh_api("GET","/git/ref/heads/main")["object"]["sha"]
original=read_artworks(BASE,"ARTWORKS",head)
extra=read_artworks(EXTRA,"NEW_ARTWORKS",head)
research=json.loads(repo_text(RESEARCH,head) or "[]")
existing=original+extra
titles_seen={normalized(x.get("title")) for x in existing}
pairs_seen={(normalized(x.get("artist")),normalized(x.get("title"))) for x in existing}
ids_seen={str(x.get("id","")).lower() for x in existing}
urls_seen={str(x.get("sourceUrl","")) for x in existing}
media_seen={str(x.get("sourceImageUrl","")).split("?")[0] for x in existing}
print("Existing artworks:",len(original),"+",len(extra))
resolved={}
for offset in range(0,len(FAMOUS_TITLES),35):
    batch=FAMOUS_TITLES[offset:offset+35]
    d=get_json(WIKI,{"action":"query","format":"json","formatversion":"2","redirects":1,"prop":"pageprops","ppprop":"wikibase_item","titles":"|".join(batch)})
    for page in d.get("query",{}).get("pages",[]):
        q=page.get("pageprops",{}).get("wikibase_item")
        if q:resolved.setdefault(q,page["title"])
art=entities(list(resolved))
artist_ids=sorted({qid for q,item in art.items() if (qid:=ent_id(first_claim(item,"P170")))})
artists=entities(artist_ids)
print("Wikipedia/Wikidata candidates:",len(art))
selected=[]; blobs={}; provenance=[]; skips=Counter(); seen_sha=set()
consecutive_blocks=0
priority={normalized(title):i for i,title in enumerate(FAMOUS_TITLES)}
def priority_key(entry):
    title=normalized(entry[1])
    return priority.get(title, next((i for t,i in priority.items() if t in title or title in t),9999))
for qid,wiki_title in sorted(resolved.items(),key=priority_key):
    if len(selected)>=TARGET:break
    item=art.get(qid,{})
    filename=first_claim(item,"P18")
    maker=artists.get(ent_id(first_claim(item,"P170")),{})
    title=label(item)
    artist=label(maker)
    yr=year(first_claim(item,"P571"))
    died=year(first_claim(maker,"P570"))
    if not (isinstance(filename,str) and title and artist and yr and 1200<=yr<=1929 and died and died<=date.today().year-71):
        skips["metadata-or-copyright-check"]+=1
        continue
    ident="wd-"+qid.lower()
    if ident in ids_seen or normalized(title) in titles_seen or normalized(wiki_title) in titles_seen or (normalized(artist),normalized(title)) in pairs_seen:
        skips["already-in-collection"]+=1
        continue
    try:meta=image_meta(filename)
    except Exception as e:
        skips["commons-api-error"]+=1
        print("Commons metadata error:",str(e)[:100])
        if "403" in str(e) or "429" in str(e):
            raise RuntimeError("Commons API access blocked; stopping instead of retrying all artworks") from e
        continue
    if not meta:
        skips["rights-not-verified"]+=1
        continue
    if meta["sha1"] in seen_sha or meta["url"].split("?")[0] in media_seen:
        skips["duplicate-file"]+=1
        continue
    try:
        with api.get(meta["url"],stream=True,timeout=90) as resp:
            if resp.status_code in (403,429):
                consecutive_blocks+=1
                print("Commons download refused",resp.status_code,":",title)
                if consecutive_blocks>=3:raise RuntimeError("Repeated image access blocks. Stopping without publishing.")
                continue
            resp.raise_for_status()
            raw=bytearray()
            for piece in resp.iter_content(65536):
                raw.extend(piece)
                if len(raw)>30*1024*1024:raise ValueError("Image >30MB")
        (game,dims),(thumb,_)=convert_image(bytes(raw))
        consecutive_blocks=0
    except RuntimeError:raise
    except Exception as e:
        skips["image-error"]+=1
        print("Image failed:",title,str(e)[:80])
        continue
    en,ja=bilingual(title,artist,yr)
    source="https://www.wikidata.org/wiki/"+qid
    record={"id":ident,"title":title,"artist":artist,"artistSort":normalized(artist),"year":str(yr),"yearStart":yr,"yearEnd":yr,"era":era(yr),"theme":theme(title),"medium":"Painting","imageUrl":"/artworks/"+ident+".webp","thumbnailUrl":"/artworks/"+ident+"-thumb.webp","width":dims[0],"height":dims[1],"museum":"Wikimedia Commons","sourceUrl":source,"license":meta["license"],"licenseUrl":meta["licenseUrl"],"licenseBasis":"Wikimedia Commons file-level Public Domain metadata checked "+date.today().isoformat(),"sourceImageUrl":meta["url"],"verifiedAt":date.today().isoformat(),"description":en,"descriptionTranslations":{"ja":ja},"descriptionVersion":"famous-flex-en-ja-v1","descriptionKind":"metadata_summary"}
    selected.append(record)
    provenance.append({"id":ident,"wikidataId":qid,"sourceUrl":source,"commonsFileUrl":meta["page"],"commonsSha1":meta["sha1"],"imageRights":{"license":meta["license"],"licenseUrl":meta["licenseUrl"],"verifiedAt":date.today().isoformat(),"authorDied":died},"originalImageDimensions":{"width":dims[0],"height":dims[1]},"manualVisualReview":"required before public release"})
    blobs["public/artworks/"+ident+".webp"]=game
    blobs["public/artworks/"+ident+"-thumb.webp"]=thumb
    ids_seen.add(ident);titles_seen.add(normalized(title));seen_sha.add(meta["sha1"]);media_seen.add(meta["url"].split("?")[0])
    print(f'Accepted {len(selected)}/{TARGET}: {title}')
print("Selected:",len(selected),"skip reasons:",dict(skips))
if not selected:raise RuntimeError("No eligible works. No GitHub changes.")
# Final recheck against the current HEAD prevents two concurrent runs from duplicating data.
now=gh_api("GET","/git/ref/heads/main")["object"]["sha"]
if now!=head:raise RuntimeError("Repository updated during collection; rerun to avoid lost updates.")
paths=gh_api("GET","/git/trees/"+gh_api("GET","/git/commits/"+head)["tree"]["sha"]+"?recursive=1")
if paths.get("truncated"):raise RuntimeError("GitHub tree truncated; no commit")
existing_paths={v["path"] for v in paths["tree"]}
if any(p in existing_paths for p in blobs):raise RuntimeError("Image path collision; no commit")
manifest="import type { Artwork } from '../types';\n\n// Generated by ART_SLIDE_World_Famous_100.ipynb\nexport const NEW_ARTWORKS: Artwork[] = "+json.dumps(extra+selected,ensure_ascii=False,indent=2)+";\n"
all_research={x.get("id"):x for x in research if isinstance(x,dict) and x.get("id")}
all_research.update({x["id"]:x for x in provenance})
blobs[EXTRA]=manifest.encode("utf-8")
blobs[RESEARCH]=(json.dumps(list(all_research.values()),ensure_ascii=False,indent=2)+"\n").encode("utf-8")
tree=[]
for i,(path,data) in enumerate(sorted(blobs.items()),1):
    sha=gh_api("POST","/git/blobs",{"content":base64.b64encode(data).decode(),"encoding":"base64"})["sha"]
    tree.append({"path":path,"mode":"100644","type":"blob","sha":sha})
    if i%40==0:print("Uploaded",i,"files")
parent_tree=gh_api("GET","/git/commits/"+head)["tree"]["sha"]
new_tree=gh_api("POST","/git/trees",{"base_tree":parent_tree,"tree":tree})["sha"]
commit=gh_api("POST","/git/commits",{"message":"Add famous public-domain paintings from Wikimedia Commons","tree":new_tree,"parents":[head]})["sha"]
gh_api("PATCH","/git/refs/heads/main",{"sha":commit,"force":False})
print("COMMITTED",len(selected),"works:", "https://github.com/"+REPO+"/commit/"+commit)


収集後、表示されたGitHubコミットURLから追加件数と画像を確認してください。既存作品は変更しません。